# Custom CNN for Palmprint Recognition

**Project:** Deep Learning-Based Palmprint Recognition System  
**Model:** Custom convolutional neural network (PyTorch)

This notebook trains a CNN to identify one of 600 palms. It trains and validates on images from `session1`, then evaluates once on the matching palms in `session2` to measure cross-session performance.

## Dataset layout

Set `DATA_ROOT` below to the folder containing `session1/` and `session2/`. Each folder can contain the original TIFF images or the 128×128 PNG crops created by `SE4050_preprocessing.ipynb`.

The dataset names samples `00001`–`00010` for palm 1, `00011`–`00020` for palm 2, and so on. The notebook derives each identity from the filename. It keeps session 2 out of training and validation.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import random
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

## 2. Find the dataset and create labels

By default, the notebook checks common project locations. If the data is elsewhere, set `DATA_ROOT` explicitly to its parent folder.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change this if your data is stored in another location.
DATA_ROOT = None
candidates = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "processed",
    PROJECT_ROOT / "data",
]
if DATA_ROOT is None:
    DATA_ROOT = next(
        (candidate for candidate in candidates
         if (candidate / "session1").is_dir() and (candidate / "session2").is_dir()),
        None,
    )
else:
    DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()

if DATA_ROOT is None:
    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find session1/ and session2/. Set DATA_ROOT to the directory "
        "containing those two folders. Checked:\n" + checked
    )

DATA_ROOT = Path(DATA_ROOT)
print("Dataset root:", DATA_ROOT)
print("Session folders:", (DATA_ROOT / "session1").is_dir(), (DATA_ROOT / "session2").is_dir())

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
NUM_CLASSES = 600

def collect_session(session_name):
    folder = DATA_ROOT / session_name
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing session folder: {folder}")
    rows = []
    for image_path in sorted(folder.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        try:
            image_number = int(image_path.stem)
        except ValueError:
            continue
        if not 1 <= image_number <= NUM_CLASSES * 10:
            continue
        # Ten consecutive image numbers belong to each palm identity.
        label = (image_number - 1) // 10
        rows.append({"path": str(image_path), "label": label, "palm_id": label + 1, "session": session_name})
    return pd.DataFrame(rows)

session1_df = collect_session("session1")
session2_df = collect_session("session2")
print("Session 1 images:", len(session1_df), "| identities:", session1_df.label.nunique())
print("Session 2 images:", len(session2_df), "| identities:", session2_df.label.nunique())
display(session1_df.head())

In [ ]:
if session1_df.empty or session2_df.empty:
    raise ValueError("No supported image files found. Check DATA_ROOT and the session folder contents.")

expected_labels = set(range(NUM_CLASSES))
for name, frame in (("session1", session1_df), ("session2", session2_df)):
    found = set(frame["label"].unique())
    missing = sorted(expected_labels - found)
    if missing:
        raise ValueError(
            f"{name} is missing {len(missing)} of the expected 600 identities "
            f"(first missing palm IDs: {[x + 1 for x in missing[:10]]}). "
            "Use the full dataset or check preprocessing failures."
        )

counts = session1_df["label"].value_counts()
if (counts < 2).any():
    raise ValueError("Each session1 identity needs at least two images for a train/validation split.")
print("Images per identity in session1:", counts.describe().round(2).to_dict())
print("Images per identity in session2:", session2_df.label.value_counts().describe().round(2).to_dict())

## 3. Train/validation split and image pipeline

The validation split comes only from `session1`. `session2` is reserved for final evaluation. Images are converted to grayscale and resized to 128×128; augmentation is applied only to training images.

In [ ]:
train_df, val_df = train_test_split(
    session1_df,
    test_size=0.20,
    random_state=SEED,
    stratify=session1_df["label"],
)

IMAGE_SIZE = 128
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])

class PalmDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("L")
            image = self.transform(image)
        return image, self.labels[index]

train_dataset = PalmDataset(train_df, train_transform)
val_dataset = PalmDataset(val_df, eval_transform)
test_dataset = PalmDataset(session2_df, eval_transform)

# num_workers=0 keeps loading reliable in Windows notebooks.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

images, labels = next(iter(train_loader))
print("Train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Batch shape:", tuple(images.shape), "| label range:", int(labels.min()), "to", int(labels.max()))

## 4. Define the custom CNN

The network uses four convolution blocks, batch normalization, max pooling, global average pooling, and dropout before the 600-way classifier.

In [ ]:
class CustomPalmCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            self._block(1, 32),
            self._block(32, 64),
            self._block(64, 128),
            self._block(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    @staticmethod
    def _block(in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

model = CustomPalmCNN().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
EPOCHS = 15
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RESULTS_DIR / "custom_cnn_best.pth"

print("Epochs:", EPOCHS)
print("Checkpoint:", CHECKPOINT_PATH)

## 5. Train and select the best validation checkpoint

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(DEVICE, non_blocking=True)
        batch_labels = batch_labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(batch_images)
            loss = criterion(logits, batch_labels)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = batch_labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total_items += batch_size

    return total_loss / total_items, total_correct / total_items

history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
best_val_accuracy = -1.0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step(val_accuracy)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save({
            "model_state_dict": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "image_size": IMAGE_SIZE,
            "best_val_accuracy": best_val_accuracy,
        }, CHECKPOINT_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_accuracy:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_accuracy:.4f} | "
        f"{time.time() - start:.1f}s"
    )

print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print("Saved best checkpoint:", CHECKPOINT_PATH)

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["train_loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["train_accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Final evaluation on session 2

Load the checkpoint with the best validation accuracy and report top-1 and top-5 identification accuracy on the held-out second session.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_targets = []
all_predictions = []
all_probabilities = []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        logits = model(batch_images.to(DEVICE, non_blocking=True))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probabilities)
        all_predictions.extend(probabilities.argmax(axis=1).tolist())
        all_targets.extend(batch_labels.numpy().tolist())

all_probabilities = np.concatenate(all_probabilities, axis=0)
test_top1 = accuracy_score(all_targets, all_predictions)
test_top5 = top_k_accuracy_score(all_targets, all_probabilities, k=5, labels=np.arange(NUM_CLASSES))
print(f"Session 2 top-1 accuracy: {test_top1:.4f}")
print(f"Session 2 top-5 accuracy: {test_top5:.4f}")
print(f"Best session 1 validation accuracy: {checkpoint['best_val_accuracy']:.4f}")

In [ ]:
# Inspect per-palm performance without printing a 600-row report.
report = classification_report(
    all_targets,
    all_predictions,
    labels=np.arange(NUM_CLASSES),
    output_dict=True,
    zero_division=0,
)
per_class = pd.DataFrame.from_dict(report, orient="index")
per_class = per_class.loc[[str(label) for label in range(NUM_CLASSES)], ["precision", "recall", "f1-score", "support"]]
per_class.index = [f"palm_{int(label) + 1:03d}" for label in per_class.index]
print("Lowest-accuracy palms:")
display(per_class.sort_values("recall").head(10))

## 7. Save a compact experiment summary

In [ ]:
summary = {
    "model": "CustomPalmCNN",
    "classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "train_images": len(train_dataset),
    "validation_images": len(val_dataset),
    "test_images": len(test_dataset),
    "best_validation_accuracy": float(checkpoint["best_val_accuracy"]),
    "session2_top1_accuracy": float(test_top1),
    "session2_top5_accuracy": float(test_top5),
    "checkpoint": str(CHECKPOINT_PATH),
}
summary_path = RESULTS_DIR / "custom_cnn_metrics.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))